# Concept construction: pooled vs hierarchical

Comparative experiment for the two `subspace/` concept builds, from the same
loaded model, via the per-call `method` override:

- **pooled** (`method="pooled"`) — every token across every lemma is pooled into
  one cloud, top-`r` SVD. One vote **per token**.
- **hierarchical** (`method="hierarchical"`) — each lemma becomes its own
  `r_lemma`-dim subspace, then the concept is the principal subspace of that set
  (top-`r` eigenvectors of `Σᵢ BᵢBᵢᵀ`). One vote **per lemma**.

Both produce the same `(d, r)` orthonormal basis, so projection/generation
scoring is identical — only the build differs. We compare on (1) expression /
energy-fraction and (2) guided generation. CPU math is in
`tests/test_subspace.py`; this is the model-in-the-loop counterpart.

In [ ]:
import sys
from pathlib import Path

# repo root is two levels up from subspace/notebooks/, so `subspace`/`frames` import
sys.path.insert(0, str(Path.cwd().parents[1]))

In [ ]:
import pandas as pd
import torch

from subspace import SubspaceUnembeddingRepresentation

In [ ]:
RANK = 8
R_LEMMA = 1  # directions kept per lemma in the hierarchical build
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24

# One model; both constructions are built from it via the `method` override.
sur = SubspaceUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    rank=RANK,
    r_lemma=R_LEMMA,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

In [ ]:
CONCEPTS = ["dog.n.01", "war.n.01"]


def build_pair(synset):
    """Build the same concept both ways from the shared model."""
    pooled = sur.get_concept(synset, MIN_LEMMAS, MAX_TOKENS, method="pooled")
    hier = sur.get_concept(synset, MIN_LEMMAS, MAX_TOKENS, method="hierarchical")
    return pooled, hier


pairs = {syn: build_pair(syn) for syn in CONCEPTS}
for syn, (pooled, hier) in pairs.items():
    print(syn, "| pooled:", pooled, "| hierarchical:", hier)

## Metric 1 — expression / energy-fraction

Mean per-position energy fraction (`cos²θ`) of each probe sentence within the
concept subspace. On-concept probes should score higher than the neutral one for
**both** builds; the interesting cell is whether hierarchical widens or narrows
the on/off gap.

In [ ]:
PROBES = {
    "dog (on)": "The puppy barked and wagged its tail at the hound",
    "war (on)": "The army launched its assault and the soldiers fought the battle",
    "neutral": "The committee reviewed the quarterly budget report",
}

rows = []
for syn, (pooled, hier) in pairs.items():
    for probe_name, text in PROBES.items():
        rows.append(
            {
                "concept": syn,
                "probe": probe_name,
                "pooled": sur.expression(text, pooled).item(),
                "hierarchical": sur.expression(text, hier).item(),
            }
        )

expr = pd.DataFrame(rows)
expr["Δ (hier - pooled)"] = expr["hierarchical"] - expr["pooled"]
expr

## Metric 2 — guided generation

Same top-k guided decoding for both builds; compare the final cumulative
alignment (`probe[:, -1]`) and the decoded text.

In [ ]:
PROMPTS = ["Tell me a story about", "My favorite thing is"]

for syn, (pooled, hier) in pairs.items():
    print(f"\n================ {syn} ================")
    for method_name, concept in (("pooled", pooled), ("hierarchical", hier)):
        texts, probe = sur.generate_with_topk_guide(
            PROMPTS, guide=concept, k=K, steps=STEPS
        )
        print(f"\n--- {method_name} ---")
        for i, text in enumerate(texts):
            print(f"  score={probe[i, -1].item():.3f} | {text}")

## Notes

- Both arms share scoring (`‖Bᵀh‖`) and rank `r`; any difference is purely the
  concept construction.
- `R_LEMMA` (directions per lemma) and `RANK` are the two knobs to sweep; the
  balancing effect of the hierarchical build is strongest at small `R_LEMMA`.
- Still SM (single-vector query). MSM (subspace query + canonical angles) is a
  separate follow-up.